# 03 — Uplift / Counterfactual Model

METIS buildathon ML training notebook. Uses the same training logic as `train_metis_models.py`.

Uplift model: T-learner bundle. The control model estimates natural recovery; each treatment model estimates recovery under a specific action. Uplift is `P(pay | action) - P(pay | control)`, so negative uplift remains possible.

In [ ]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, log_loss, roc_auc_score
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
BASE_FEATURES = [
    "amount", "days_overdue", "previous_failed_count",
    "historical_payment_rate", "contact_count",
    "previous_no_response_count", "segment_high_value",
    "segment_at_risk", "failure_insufficient_funds",
]
INTERVENTIONS = ["RETRY", "REMINDER", "PAYMENT_LINK", "PAYMENT_PLAN", "NEGOTIATION"]

ROOT = Path.cwd()
for candidate in [
    ROOT / "recovery_events.csv",
    ROOT / "../recovery_events.csv",
    ROOT / "../data/recovery_events.csv",
    ROOT / "../../ml_data/recovery_events.csv",
]:
    if candidate.exists():
        DATA_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError("Place recovery_events.csv in the notebook directory, ../, ../data/, or ../../ml_data/")

OUTPUT_DIR = ROOT / "../backend/models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)


In [ ]:

def feature_frame(df):
    out = pd.DataFrame(index=df.index)
    for name in [
        "amount", "days_overdue", "previous_failed_count",
        "historical_payment_rate", "contact_count",
        "previous_no_response_count"
    ]:
        out[name] = pd.to_numeric(df[name], errors="coerce").fillna(0.0)

    out["segment_high_value"] = (
        df["segment"].fillna("REGULAR").astype(str).str.upper() == "HIGH_VALUE"
    ).astype(float)
    out["segment_at_risk"] = (
        df["segment"].fillna("REGULAR").astype(str).str.upper() == "AT_RISK"
    ).astype(float)
    out["failure_insufficient_funds"] = (
        df["failure_reason"].fillna("").astype(str).str.lower().str.contains("insufficient")
    ).astype(float)
    return out[BASE_FEATURES].astype(float)

def make_classifier(seed=RANDOM_STATE):
    return LGBMClassifier(
        objective="binary",
        n_estimators=260,
        learning_rate=0.035,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.60,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )


In [ ]:

prop_path = OUTPUT_DIR / "propensity_model.pkl"
int_path = OUTPUT_DIR / "intervention_model.pkl"

if not prop_path.exists() or not int_path.exists():
    raise FileNotFoundError(
        "Run 01_propensity_model.ipynb and 02_intervention_response_models.ipynb first."
    )

prop_bundle = joblib.load(prop_path)
int_bundle = joblib.load(int_path)

uplift_bundle = {
    "control_model": prop_bundle["model"],
    "treatment_models": int_bundle["models"],
    "feature_names": BASE_FEATURES,
    "model_version": "metis-uplift-tlearner-v1",
    "method": "T_LEARNER",
}

joblib.dump(uplift_bundle, OUTPUT_DIR / "uplift_model.pkl", compress=3)
print("Saved:", OUTPUT_DIR / "uplift_model.pkl")


In [ ]:

# Offline policy sanity-check against hidden simulator potential outcomes.
# These hidden columns are ONLY for evaluation and never used for training.
_, eval_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["intervention_type"]
)
eval_df = df.iloc[eval_idx].copy()
X = feature_frame(eval_df)

p0 = uplift_bundle["control_model"].predict_proba(X)[:, 1]
predicted = []
truth = []

for action in INTERVENTIONS:
    pa = uplift_bundle["treatment_models"][action].predict_proba(X)[:, 1]
    predicted.append(pa - p0)

    truth.append(
        eval_df[f"_p_{action.lower()}_truth"].to_numpy()
        - eval_df["_p_control_truth"].to_numpy()
    )

predicted = np.column_stack(predicted)
truth = np.column_stack(truth)

best_pred = predicted.argmax(axis=1)
best_true = truth.argmax(axis=1)

metrics = {
    "mean_absolute_uplift_error": float(np.mean(np.abs(predicted - truth))),
    "mean_selected_true_uplift": float(np.mean(truth[np.arange(len(truth)), best_pred])),
    "mean_oracle_uplift": float(np.mean(truth[np.arange(len(truth)), best_true])),
    "best_action_accuracy": float(np.mean(best_pred == best_true)),
}
metrics


In [ ]:

with open(OUTPUT_DIR / "uplift_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)
